<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Complaints Clustering Using Teradata VantageCloud and open-source language models
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:20px;font-family:Arial;'><b>Introduction:</b></p>

<p style="font-size:16px;font-family:Arial;">This feature uses advanced clustering techniques powered by <b>Teradata Vantage</b> and <b>Open source Embeddings</b> model to group similar customer complaints together. By identifying common themes and patterns, this functionality provides valuable insights into the key issues and pain points experienced by customers.</p>


<p style="font-size:16px;font-family:Arial;"><b>Key Features of Complaints Clustering:</b></p>
<ul style="font-size:16px;font-family:Arial;">
  <li>Leverages advanced clustering algorithms powered by <b>Teradata Vantage</b> and <b>Open source Embeddings.</b></li>
  <li>Groups similar customer complaints together, revealing common themes and pain points.</li>
  <li>Provides clients with a deeper understanding of the key issues affecting their customers.</li>
  <li>Enables clients to prioritize and address the most pressing concerns more effectively.</li>
  <li>Helps clients identify opportunities for product improvements and enhanced customer experience.</li>
</ul>


<p style = 'font-size:16px;font-family:Arial;'>Unlock the revolutionary potential of Generative AI to categorize and analyze complaints with unparalleled efficiency.</p>

<p style = 'font-size:16px;font-family:Arial;'><b>Steps in the analysis:</b></p>
<ol style = 'font-size:16px;font-family:Arial;'>
  <li>Configuring the environment</li>
  <li>Connect to Vantage</li>
  <li>Create a Custom Container in Vantage</li>
  <li>Install Dependencies</li>
  <li>Operationalizing AI-powered analytics</li>
  <li>Complaints Analysis</li>
  <li>Cluster the Complaints</li>
  <li>Cleanup</li>
</ol>

<hr style='height:2px;border:none;'>
<b style = 'font-size:20px;font-family:Arial;'>1. Configuring the environment</b>

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>1.1 Install the required libraries</b></p>

In [ ]:
%%capture

!pip install -r requirements2.txt --quiet

In [ ]:
# !pip install teradataml==20.0.0.7 teradatamlwidgets==20.0.0.6 teradatamodelops==7.0.3 teradatasql==20.0.0.34 teradatasqlalchemy==20.0.0.7 sentencepiece sentence-transformers wordcloud --quiet

<div class="alert alert-block alert-info">
<p style = 'font-size:16px;font-family:Arial;'><b>Note: </b><i>Please restart the kernel after executing these two lines. The simplest way to restart the Kernel is by typing zero zero: <b> 0 0</b></i></p>

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>1.2 Import the required libraries</b></p>
<p style = 'font-size:16px;font-family:Arial;'>Here, we import the required libraries, set environment variables and environment paths (if required).</p>

In [ ]:
# Standard library imports
import csv
import os
import sys
import time
import warnings
from collections import OrderedDict
from os.path import expanduser

# Third-party imports
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from wordcloud import WordCloud

# Teradata imports
from teradataml import *
from teradatasqlalchemy.types import *

# IPython display utilities
from IPython.display import Markdown, clear_output, display
from IPython.display import display as ipydisplay

import plotly.io as pio
pio.renderers.default = 'notebook'

%matplotlib inline

# Suppress warnings and Vantage runtime messages
warnings.filterwarnings('ignore')
display.suppress_vantage_runtime_warnings = True

# Pandas display settings for better readability
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 100)

# Append config utilities path and import helpers
module_path = os.path.abspath(os.path.join('..', '..', 'config'))
sys.path.append(module_path)
from utils.oaf_utils import *

# Python version for the OAF container (must match the base image)
PYTHON_VERSION = "3.11"
print(f'Using Python version {PYTHON_VERSION} for user environment')

# Hugging Face sentence-transformer model used for generating embeddings
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Required packages to install in the custom OAF container
REQUIRED_PACKAGES = [
    'numpy',
    'transformers',
    'torch',
    'sentencepiece',
    'pandas',
    'sentence-transformers',
    'dill',
]

# Container name for the Open Analytics Framework environment
OAF_NAME = 'oaf_cluster'

<hr style="height: 2px; border: none;">
<p style = 'font-size: 20px; font-family: Arial;'><b>Part 1</b></p>

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial;'>2. Connect to Vantage</b>

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial;'><b>2.1 Connect to Vantage</b></p>
<p style = 'font-size:16px;font-family:Arial;'>Load the environment variables from a .env file and use them to create a connection context to Teradata.</p>

In [ ]:
# Load database credentials from environment file
print("Creating the context...")
load_dotenv("../../.config/.env", override=True)
host = os.getenv("host")
username = os.getenv("username")
password = os.getenv("my_variable")

# Establish connection to Teradata Vantage
eng = create_context(host=host, username=username, password=password)

# Tag the session for tracking/debugging purposes
execute_sql(
    """SET query_band='DEMO=Complaints_Clustering_VCL.ipynb;' UPDATE FOR SESSION;"""
)
print("Connected to Teradata:", eng)

<p style = 'font-size:16px;font-family:Arial;'>Begin running steps with Shift + Enter keys. </p>

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>2.2  Connect to the Environment Service</b></p>

<p style = 'font-size:16px;font-family:Arial;'>To better support integration with Cloud Services and common automation tools; the <b > User Environment Service</b> is accessed via RESTful APIs.  These APIs can be called directly or in the examples shown below that leverage the Python Package for Teradata (teradataml) methods.</p> 

In [ ]:
# Verify the environment configuration file exists before proceeding
ENV_CONFIG_PATH = "/home/jovyan/JupyterLabRoot/VantageCloud_Lake/.config/.env"

if os.path.exists(ENV_CONFIG_PATH):
    print("Your environment parameter file exists. Please proceed with this use case.")
    env_vars = dotenv_values(ENV_CONFIG_PATH)
else:
    print("Your environment has not been prepared for connecting to VantageCloud Lake.")
    print("Please contact the support team.")

In [ ]:
# Retrieve compute group for GPU-based processing
compute_group = os.getenv("gpu_compute_group")

# Authenticate with the User Environment Service (UES)
if set_auth_token(
    base_url=env_vars.get("ues_uri"),
    pat_token=env_vars.get("access_token"),
    pem_file=env_vars.get("pem_file"),
    valid_from=int(time.time()),
):
    print("UES Authentication successful")
else:
    print("UES Authentication failed. Check credentials.")
    sys.exit(1)

<p style = 'font-size:16px;font-family:Arial;'>After connecting and authenticating, check cluster status. Start it if necessary - note the cluster only needs to be running to execute the APPLY sections of the demo.</p>

In [ ]:
# Set the compute group for this session and ensure the cluster is running
execute_sql(f"SET SESSION COMPUTE GROUP {compute_group};")
res = check_cluster_start(compute_group=compute_group)

In [ ]:
list_user_envs()

<hr style="height:2px;border:none;">

<b style = 'font-size:20px;font-family:Arial;'>3. Create a Custom Container in Vantage</b>

<p style = 'font-size:16px;font-family:Arial;'>If desired, the user can create a <b>new</b> custom environment by starting with a "base" image and customizing it.  The steps are:</p> 
<ul style = 'font-size:16px;font-family:Arial;'>
    <li>List the available "base" images the system supports</li>
    <li>List any existing "custom" environments the user has created</li>
    <li>If there are no custom environments, then create a new one from a base image</li>
    </ul>

In [ ]:
# List existing user environments (if any)
try:
    ipydisplay(list_user_envs())
except Exception as e:
    if "No user environments found" in str(e):
        print("No user environments found")
    else:
        raise

print(f"OAF Environment is set to: {OAF_NAME}")

# Create a new environment or connect to an existing one
try:
    demo_env = create_env(
        env_name=OAF_NAME,
        base_env=f"python_{PYTHON_VERSION}",
        desc="OAF Demo env for LLM",
    )
except Exception as e:
    error_msg = str(e)
    if "same name already exists" in error_msg:
        print("Environment already exists, obtaining a reference to it")
        demo_env = get_env(OAF_NAME)
    elif "Invalid value for base environment name" in error_msg:
        print("Unsupported base environment version, using defaults")
        demo_env = create_env(env_name=OAF_NAME, desc="OAF Demo env for LLM")
    else:
        raise

# Wait for environment creation to register (asynchronous operation)
sleep(5)

# Verify environment is listed
try:
    ipydisplay(list_user_envs())
except Exception as e:
    if "No user environments found" in str(e):
        print("No user environments found")
    else:
        raise

<hr style='height:2px;border:none;'>

<b style = 'font-size:20px;font-family:Arial;'> 4. Install Dependencies</b>

<p style = 'font-size:16px;font-family:Arial;'>The second step in the customization process is to install Python package dependencies. This demonstration uses the Hugging Face <a href = 'https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2'>all-MiniLM-L6-v2</a> Sentence Transformer.  Since VantageCloud Lake Analytic Clusters are secured by default against unauthorized access to the outside network, the user can load the required libraries and model using teradataml methods:
</p> 

<ul style = 'font-size:16px;font-family:Arial;'>
    <li>List the currently installed models and python libraries</li>
    <li><b>If necessary</b>, install any required packages</li>
    <li><b>If necessary</b>, install the pre-trained model.  This process takes several steps;
        <ol style = 'font-size:16px;font-family:Arial;'>
            <li>Import and download the model</li>
            <li>Create a zip archive of the model artifacts</li>
            <li>Call the install_model() method to load the model to the container</li>
        </ol></li>
    </ul>

In [ ]:
# Display currently installed models and a sample of installed libraries
ipydisplay(demo_env.models)
ipydisplay(demo_env.libs.head(5))

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>4.1 A note on package versions</b></p>

<p style = 'font-size:16px;font-family:Arial;'>The next demonstration makes use of the DataFrame apply() method, which automatically passes the python code to the Analytic Cluster.  As such, one needs to ensue the python package versions match.  dill and pandas are required, as is any additional libraries for the use case.
</p> 

<p style = 'font-size:16px;font-family:Arial;'><b>Note</b> while not required for many OAF use cases, for this demo the required packages for the model execution must be installed in the local environment first.</p>

In [ ]:
import importlib


def get_versioned_packages(package_names):
    """Retrieve version-pinned package strings for local packages.

    Returns a list of strings like ['numpy==1.24.0', 'torch==2.0.1', ...].
    """
    versioned = []
    for pkg in package_names:
        module_name = pkg.replace("-", "_")
        mod = importlib.import_module(module_name)
        versioned.append(f"{pkg}=={mod.__version__}")
    return versioned


versioned_pkgs = get_versioned_packages(REQUIRED_PACKAGES)

# Check if packages are already installed by comparing required vs. installed names
installed_names = set(demo_env.libs["name"].to_list())
required_names = {pkg.split("==")[0] for pkg in REQUIRED_PACKAGES}

if not required_names.issubset(installed_names):
    # Strip build metadata (e.g. '+cu118') from version strings before installing
    clean_pkgs = [v.split("+")[0] for v in versioned_pkgs]
    claim_id = demo_env.install_lib(clean_pkgs, asynchronous=True)
else:
    print(f"All required packages are installed in the {OAF_NAME} environment")

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>4.2 Monitor library installation status</b></p>

<p style = 'font-size:16px;font-family:Arial;'>Optionally - users can monitor the library installation status using the cell below:
</p> 

In [ ]:
# Monitor library installation progress (polls until complete)
try:
    claim_id  # Check if an installation was triggered
    ipydisplay(demo_env.status(claim_id))

    stage = demo_env.status(claim_id)["Stage"].iloc[-1]
    while stage == "Started":
        stage = demo_env.status(claim_id)["Stage"].iloc[-1]
        clear_output()
        ipydisplay(demo_env.status(claim_id))
        sleep(5)
except NameError:
    print("No installations to monitor")

# Verify installed libraries
ipydisplay(demo_env.libs)

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>4.3 Download and install model</b></p>

<p style = 'font-size:16px;font-family:Arial;'>Open Analytics Framework containers do not have open access to the external network, which contributes to a very secure runtime environment.  As such, users will load pre-trained models using the below APIs.  For illustration purposes, the following code will check to see if the model archive exists locally and if it doesn't, will import and download it by creating a model object.  The archive will then be created and installed into the remote environment.
</p> 

In [ ]:
# Construct the local archive filename from the model name
# e.g. "sentence-transformers/all-MiniLM-L6-v2" -> "models--sentence-transformers--all-MiniLM-L6-v2"
model_archive_name = "models--" + MODEL_NAME.replace("/", "--")
model_archive_path = f"{model_archive_name}.zip"
print(f"Model archive: {model_archive_name}")

# Download model and create archive if it doesn't exist locally
if not os.path.isfile(model_archive_path):
    from sentence_transformers import SentenceTransformer
    import shutil

    print("Downloading model and creating archive...")
    model = SentenceTransformer(MODEL_NAME)

    # Archive the cached model files into a zip for upload
    cache_dir = f'{expanduser("~")}/.cache/huggingface/hub/{model_archive_name}/'
    shutil.make_archive(model_archive_name, format="zip", root_dir=cache_dir)
else:
    print("Local model archive already exists.")

# Install the model into the OAF container if not already present
try:
    models_df = demo_env.models
    model_installed = (
        models_df is not None
        and not models_df.empty
        and any(model_archive_name in m for m in models_df["Model"])
    )

    if model_installed:
        print("Model already installed")
    else:
        print("Installing model...")
        claim_id = demo_env.install_model(
            model_path=model_archive_path, asynchronous=True
        )
except (AttributeError, TypeError):
    # Handle cases where demo_env.models is None or unexpected type
    print("Installing model...")
    claim_id = demo_env.install_model(model_path=model_archive_path, asynchronous=True)

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>4.4 Monitor model installation status</b></p>

<p style = 'font-size:16px;font-family:Arial;'>Optionally - users can monitor the model installation status using the cell below:
</p> 

In [ ]:
# Monitor model installation progress (polls until "File Installed")
try:
    claim_id  # Check if an installation was triggered
    ipydisplay(demo_env.status(claim_id))

    stage = demo_env.status(claim_id)["Stage"].iloc[-1]
    while stage != "File Installed":
        stage = demo_env.status(claim_id)["Stage"].iloc[-1]
        clear_output()
        ipydisplay(demo_env.status(claim_id))
        sleep(5)
except NameError:
    print("No installations to monitor")

# Refresh and verify the model is installed
demo_env.refresh()
ipydisplay(demo_env.models)

<p style = 'font-size:16px;font-family:Arial;'>The preceding demo showed how users can perform a <b>one-time</b> configuration task to prepare a custom environment for analytic processing at scale.  Once this configuration is complete, these containers can be re-used in ad-hoc development tasks, or used for operationalizing analytics in production.</p>

<hr style="height: 2px; border: none;">
<p style = 'font-size: 20px; font-family: Arial;'><b>Part 2</b></p>

<hr style='height:2px;border:none;'>
<b style = 'font-size:20px;font-family:Arial;'>5. Operationalizing AI-powered analytics</b>
<p style = 'font-size:16px;font-family:Arial;'>The following demonstration will illustrate how developers can take the next step in the process to <b>operationalize</b> this processing, enabling the entire organization to leverage AI across the data lifecycle, including</p>

<table style = 'width:100%;table-layout:fixed;'>
    <tr>
        <td style = 'vertical-align:top' width = '30%'>
           <ol style = 'font-size:16px;font-family:Arial;'>
               <li><b>Prepare the environment</b>.  Package the scoring function into a more robust program, and stage it on the remote environment</li>
            <br>
            <br>
               <li><b>Python Pipeline</b>.  Execute the function using Python methods</li>
            <br>
            <br>
               <li><b>SQL Pipeline</b>.  Execute the function using SQL - allowing for broad adoption and use in ETL and operational needs</li>
        </ol>
        </td>
        <td width = '20%'></td>
        <td style = 'vertical-align:top'><img src = 'images/OAF_Ops.png' width=350 style="border: 4px solid #404040; border-radius: 10px;" ></td>
    </tr>
</table>


<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>5.1 Check connection</b></p>
<p style = 'font-size:16px;font-family:Arial;'>Reconnect to the database, UES, and start cluster if necessary<get_context()/p> 

In [ ]:
# Re-establish connection if session was interrupted (e.g. kernel restart)
eng = check_and_connect(
    host=host, username=username, password=password, compute_group=compute_group
)
print(eng)

# Re-authenticate with UES
if set_auth_token(
    base_url=env_vars.get("ues_uri"),
    pat_token=env_vars.get("access_token"),
    pem_file=env_vars.get("pem_file"),
    valid_from=int(time.time()),
):
    print("UES Authentication successful")
else:
    print("UES Authentication failed. Check credentials.")
    sys.exit(1)

# Reconnect to the OAF environment and ensure cluster is running
demo_env = get_env(OAF_NAME)
check_cluster_start(compute_group=compute_group)

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>5.2 Create a server-side sentiment function</b></p>

<p style = 'font-size:16px;font-family:Arial;'>The goal of this exercise is to create a <b>server-side</b> function which can be staged on the analytic cluster.  This offers many improvements over the method used above;</p> 
<ul style = 'font-size:16px;font-family:Arial;'>
    <li><b>Performance</b>.  Staging the code and dependencies in the container environment reduces the amount of I/O, since the function doesn't need to get serialized to the cluster when called</li>
    <li><b>Operationalization</b>.  The execution pipeline can be encapsulated into a SQL statement, which allows for seamless use in ETL pipelines, dashboards, or applications that need access</li>
    <li><b>Flexibility</b>. Developers can express much greater flexibility in how the code works to optimize for performance, stability, data cleanliness or flow logic</li>
</ul>

<p style = 'font-size:16px;font-family:Arial;'>These benefits do come with some amount of additional work.  Developers need to account how data is passed in and out of the code runtime, and how to pass it back to the SQL engine to assemble and return the final resultset.  Code is executed when the user expresses an <a href = 'https://docs.teradata.com/r/Teradata-VantageCloud-Lake/SQL-Reference/SQL-Operators-and-User-Defined-Functions/Table-Operators/APPLY'>APPLY SQL function</a>;</p> 
<ol style = 'font-size:16px;font-family:Arial;'>
    <li><b>Input Query</b>.  The APPLY function takes a SQL query as input.  This query can be as complex as needed and include data preparation, cleansing, and/or any other set-based logic necessary to create the desired input data set.  This complexity can also be abstracted into a database view.  When using the teradata client connectors for Python or R, thise query is represented as a DataFrame or tibble.</li>
    <li><b>Pre-processing</b>.  Based on the query plan, data is retrieved from storage (cache, block storage, or object storage) and the input query is executed.</li>
    <li><b>Distribution</b>.  Input data can be partitioned and/or ordered to be processed on a specific container or collection of them.  For example, the user may want to process all data for a single post code in one partition, and run thousands of these in parallel.  Data can also be distributed evenly across all units of parallelism in the system</li>
    <li><b>Input</b>.  The data for each container is passed to the runtime using tandard input (stdin)</li>
    <li><b>Processing</b>.  The user's code executes, parsing stdin for the input data</li>
    <li><b>Output</b>.  Data is sent out of the code block using standard output (stdout)</li>
    <li><b>Resultset</b>.  Resultset is assembled by the analytic database, and the SQL query returns</li>
    </ol>


<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>5.3 Example server-side code block</b></p>

<p style = 'font-size:16px;font-family:Arial;'>This is the python script used in the demonstration.  It is saved to the filesystem as <code>Complaints_Clustering_OAF.py</code>.  Note here the original client-side processing function has been reused, and the additional logic is for input, output, and error handling.</p> 


<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>5.4.  Install the file and any additional artifacts</b></p>

<p style = 'font-size:16px;font-family:Arial;'>Use the install_file() method to install this python file to the container.  As a reminder, this container is persistent, so these steps need only be done infrequently.</br>
Note: Ensure that a valid .zip file path is provided in the <code>"model_path"</code> variable within the .py file below. </p> 

In [ ]:
# Stage the clustering script on the OAF container
demo_env.install_file("./library/Complaints_Clustering_OAF.py", replace=True)

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>5.5  Call the APPLY function </b></p>
<p style = 'font-size:16px;font-family:Arial;'>This function can be executed in two ways;</p> 
<ul style = 'font-size:16px;font-family:Arial;'>
    <li><b><a href = 'https://docs.teradata.com/r/Teradata-VantageCloud-Lake/Analyzing-Your-Data/Teradata-Package-for-Python-on-VantageCloud-Lake/Working-with-Open-Analytics/teradataml-Apply-Class-for-APPLY-Table-Operator'>Python</a></b> by calling the Apply() module function</li>
    <li><b><a href = 'https://docs.teradata.com/r/Teradata-VantageCloud-Lake/SQL-Reference/SQL-Operators-and-User-Defined-Functions/Table-Operators/APPLY'>SQL</a></b> which allows for broad adoption across the enterprise</li>
    </ul>
    

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>5.6 APPLY using Python</b></p>

<p style = 'font-size:16px;font-family:Arial;'>The process is as follows</p> 
<ol style = 'font-size:16px;font-family:Arial;'>
    <li>Construct a dictionary that will define the return columns and data types</li>
    <li>Construct a teradataml DataFrame representing the data to be processed - note this is a "virtual" object representing data and logic <b>in-database</b></li>
    <li>Execute the module function.  This constructs the function call in the database, but does not execute anything.  Note the Apply function takes several arguments - the input data, environment name, and the command to run</li>
    <li>In order to execute the function, an "execute_script()" method must be called.  This method returns the server-side DataFrame representing the complete operation.  This DataFrame can be used in further processing, stored as a table, etc.</li>
    </ol>
    

In [ ]:
# The all-MiniLM-L6-v2 model produces 384-dimensional embeddings
EMBEDDING_DIM = 384

# Define return column types for the APPLY function output
types_dict = OrderedDict()
types_dict["complaint_id"] = BIGINT()
types_dict["consumer_complaint_narrative"] = VARCHAR(1000)

for i in range(EMBEDDING_DIM):
    types_dict[f"embeddings_{i}"] = FLOAT()

# Build input DataFrame: select top 100 complaints with cleaned narrative text
# Replaces carriage returns, newlines, tabs, commas, and quotes with spaces
tdf = DataFrame.from_query(
    """SELECT TOP 100
        complaint_id, date_received, product, issue, sub_issue, submitted_via,
        CASE
            WHEN consumer_complaint_narrative IS NULL THEN ' '
            ELSE OREPLACE(OREPLACE(OREPLACE(OREPLACE(OREPLACE(
                consumer_complaint_narrative, X'0d', ' '), X'0a', ' '), X'09', ' '), ',', ' '), '"', ' ')
        END AS consumer_complaint_narrative
    FROM Demo_ComplaintAnalysis.Consumer_Complaints
    WHERE consumer_complaint_narrative <> '';"""
)

In [ ]:
# Configure the APPLY table operator to run the embedding script on the cluster
apply_obj = Apply(
    data=tdf[["complaint_id", "consumer_complaint_narrative"]],
    apply_command="python Complaints_Clustering_OAF.py",
    returns=types_dict,
    env_name=OAF_NAME,
    delimiter=",",
)

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>5.7 Execute the function</b></p>
<p style = 'font-size:16px;font-family:Arial;'>call execute_script(), and return a single record to the client to check the data.</p> 

In [ ]:
# Execute the server-side embedding function and display results
embeddings_df = apply_obj.execute_script()
ipydisplay(embeddings_df)

<p style = 'font-size:16px;font-family:Arial;'>Now, lets merge embeddings and row columns for further processing.</p>

In [ ]:
# Join embeddings back with the original complaint metadata
embeddings_df_merged = tdf.merge(
    right=embeddings_df,
    left_on="complaint_id",
    right_on="complaint_id",
    how="inner",
    lsuffix="tdf",
    rsuffix="emb",
)

# Drop duplicate columns introduced by the merge
embeddings_df_merged = embeddings_df_merged.drop(
    columns=["complaint_id_emb", "consumer_complaint_narrative_emb"]
)

<p style = 'font-size:16px;font-family:Arial;'>Now the results can be saved back to Vantage.</p> 

In [ ]:
# Persist the embeddings table back to Teradata Vantage
EMBEDDINGS_TABLE = "complaints_embeddings"

copy_to_sql(
    df=embeddings_df_merged,
    table_name=EMBEDDINGS_TABLE,
    if_exists="replace",
)

<p style = 'font-size:16px;font-family:Arial;'>Now, let's load full embeddings table from Vantage</p>

In [ ]:
# Load the full embeddings table as a teradataml DataFrame for analysis
complaints_embeddings_tdf = DataFrame(EMBEDDINGS_TABLE)

<hr style='height:2px;border:none;'>
<b style = 'font-size:20px;font-family:Arial;'>6. Complaints Analysis</b>
<p style = 'font-size:16px;font-family:Arial;'>We'll analyze the sample of customer complaints data.</p> 


<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>6.1 Graph for Count of Product Complaints Over Years</b></p>

<p style='font-size:16px;font-family:Arial;'>The provided graph visualizes the count of complaints over the past few years, categorized by product names.</p>

In [ ]:
# Extract the year from date_received for yearly trend analysis
viz_df = complaints_embeddings_tdf.assign(
    year=complaints_embeddings_tdf.date_received.year_of_calendar()
)

# Aggregate complaint counts by product and year
pd_df = (
    viz_df.select(["product", "year", "complaint_id_tdf"])
    .groupby(["product", "year"])
    .agg(["count"])
    .to_pandas()
)

In [ ]:
pd_df

In [ ]:
# Sort by year within each product for a clean line chart
pd_df_sorted = pd_df.sort_values(by=["product", "year"])

# Plot complaint count trends over years, colored by product
fig = px.line(
    pd_df_sorted,
    x="year",
    y="count_complaint_id_tdf",
    color="product",
    markers=True,
    title="Count of Product Complaints Over Years",
)
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Count",
    legend_title="Product",
    width=1200,
    height=600,
)
fig.show()

<hr style='height:1px;border:none;'> 
<p style='font-size:18px;font-family:Arial;'><b>6.2 Graph for Count of Complaints by Months</b></p> 
<p style='font-size:16px;font-family:Arial;'>The provided graph visualizes the count of complaints by months. We can see that the mean count is above 500, and the July and August months have the maximum complaints count.</p>

In [ ]:
# Extract the month from date_received for monthly distribution analysis
df = complaints_embeddings_tdf.assign(
    complaint_month=complaints_embeddings_tdf.date_received.month_of_year()
)

In [ ]:
# Aggregate complaint counts by month
grp_month = (
    df.select(["complaint_month", "complaint_id_tdf"])
    .groupby(["complaint_month"])
    .agg(["count"])
    .to_pandas()
)

# Map month numbers to readable names
MONTH_NAMES = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December",
}
grp_month["month"] = grp_month["complaint_month"].map(MONTH_NAMES)

# Plot monthly complaint distribution
fig = px.bar(
    grp_month.sort_values(by="complaint_month"),
    x="month",
    y="count_complaint_id_tdf",
    labels={
        "count_complaint_id_tdf": "Number of Complaints",
        "month": "Complaint Month",
    },
    title="Number of Complaints by Month",
)
fig.update_traces(hovertemplate="Month: %{x}<br>Number of Complaints: %{y:,}")
fig.show()

<hr style='height:1px;border:none;'> 

<p style='font-size:18px;font-family:Arial;'><b>6.3 Graph for Number of Complaints by Product</b></p> <p style='font-size:16px;font-family:Arial;'>The graph displays the number of complaints received for different products. The data shows that the highest number of complaints are related to credit cards or prepaid cards, as well as credit reporting and credit repair services.</p>

In [ ]:
# Aggregate complaint counts by product
grp_product = (
    df[["product", "complaint_id_tdf"]].groupby(["product"]).agg(["count"]).to_pandas()
)

fig = px.bar(
    grp_product,
    x="product",
    y="count_complaint_id_tdf",
    labels={"count_complaint_id_tdf": "Number of Complaints", "product": "Product"},
    title="Number of Complaints by Product",
)
fig.update_traces(hovertemplate="Product: %{x}<br>Number of Complaints: %{y:,}")
fig.show()

<hr style='height:1px;border:none;'> 

<p style='font-size:18px;font-family:Arial;'><b>6.4 Graph for Number of Complaints by Issue</b></p> <p style='font-size:16px;font-family:Arial;'>The graph displays the number of complaints received for different issues. The data shows that the highest number of complaints are related to issue of incorrect information on your report.</p>

In [ ]:
# Aggregate complaint counts by issue and show top 10
TOP_N = 10

grp_issue = (
    df.select(["issue", "complaint_id_tdf"])
    .groupby(["issue"])
    .agg(["count"])
    .to_pandas()
)
grp_issue = grp_issue.sort_values("count_complaint_id_tdf", ascending=False).head(TOP_N)

fig = px.bar(
    grp_issue,
    x="issue",
    y="count_complaint_id_tdf",
    labels={"count_complaint_id_tdf": "Number of Complaints", "issue": "Issue"},
    title=f"Number of Complaints by Issue (Top {TOP_N})",
)
fig.update_traces(hovertemplate="Issue: %{x}<br>Number of Complaints: %{y:,}")
fig.show()

<hr style='height:1px;border:none;'> 

<p style='font-size:18px;font-family:Arial;'><b>6.5 Graph for Number of Complaints by Sub-Issue</b></p> 

<p style='font-size:16px;font-family:Arial;'>The graph displays the number of complaints received for different sub-issues. The data shows that the highest number of complaints are related to issue of information belongs to someone else.</p>

In [ ]:
# Aggregate complaint counts by sub-issue and show top 10
grp_sub_issue = (
    df.select(["sub_issue", "complaint_id_tdf"])
    .groupby(["sub_issue"])
    .agg(["count"])
    .to_pandas()
)
grp_sub_issue = grp_sub_issue.sort_values(
    "count_complaint_id_tdf", ascending=False
).head(TOP_N)

fig = px.bar(
    grp_sub_issue,
    x="sub_issue",
    y="count_complaint_id_tdf",
    labels={"count_complaint_id_tdf": "Number of Complaints", "sub_issue": "Sub-Issue"},
    title=f"Number of Complaints by Sub-Issue (Top {TOP_N})",
)
fig.update_traces(hovertemplate="Sub-Issue: %{x}<br>Number of Complaints: %{y:,}")
fig.show()

<hr style='height:1px;border:none;'> 

<p style='font-size:18px;font-family:Arial;'><b>6.6 Graph for Number of Complaints by Channel</b></p>

<p style='font-size:16px;font-family:Arial;'>The graph displays the number of complaints received for different issues. The data shows that the all the complaints are submitted by web channel.</p>

In [ ]:
# Aggregate complaint counts by submission channel
grp_channel = (
    df.select(["submitted_via", "complaint_id_tdf"])
    .groupby(["submitted_via"])
    .agg(["count"])
    .to_pandas()
)

fig = px.bar(
    grp_channel,
    x="submitted_via",
    y="count_complaint_id_tdf",
    labels={
        "count_complaint_id_tdf": "Number of Complaints",
        "submitted_via": "Submitted Via",
    },
    title="Number of Complaints by Channel",
)
fig.update_traces(hovertemplate="Submitted Via: %{x}<br>Number of Complaints: %{y:,}")
fig.show()

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial;'>7. Cluster the Complaints</b>

<p style = 'font-size:16px;font-family:Arial;'>For our complaint clustering task, we'll be using a sample of the data to cluster the complaints. This approach will allow us to effectively analyze and categorize the complaints without using the entire dataset.</p>

In [ ]:
# Select the ID column and all embedding columns for clustering
NUM_CLUSTERS = 5
embeddings_tdf = DataFrame(EMBEDDINGS_TABLE)
embedding_columns = embeddings_tdf.columns[7:]
data_cols = embeddings_tdf.columns[:1] + embedding_columns

# Run KMeans clustering on the embedding vectors
KMeans_Model = KMeans(
    data=embeddings_tdf[data_cols],
    id_column="complaint_id_tdf",
    target_columns=embedding_columns,
    output_cluster_assignment=True,
    num_clusters=NUM_CLUSTERS,
)

In [ ]:
# Join cluster assignments back to the full embeddings table
embeddings_cluster = embeddings_tdf.join(
    other=KMeans_Model.result,
    how="inner",
    on="complaint_id_tdf=complaint_id_tdf",
    lprefix="L_",
)

In [ ]:
# Preview complaints assigned to cluster 1
embeddings_cluster[
    ["td_clusterid_kmeans", "complaint_id_tdf", "consumer_complaint_narrative_tdf"]
].loc[embeddings_cluster.td_clusterid_kmeans == 1]

<hr style='height:1px;border:none;'> 

<p style='font-size:18px;font-family:Arial;'><b>7.1 Visualization of Clusters with Complaints</b></p> 

<p style='font-size:16px;font-family:Arial;'>The graph displays the clustering of complaints into distinct groups. Based on the analysis, the data has been divided into 5 optimal clusters, each representing a unique pattern or category of complaints. This clustering approach helps to identify the key areas or types of complaints that are most prevalent, allowing for more targeted investigation and resolution efforts.</p>

In [ ]:
# Pull clustered data to local pandas DataFrame for visualization
clus = embeddings_cluster.to_pandas()

In [ ]:
# Reduce high-dimensional embeddings to 2D using t-SNE for visualization
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
tsne_result = tsne.fit_transform(clus.iloc[:, 20:-1])

In [ ]:
# Build a DataFrame with t-SNE coordinates and cluster metadata
tsne_df = pd.DataFrame(tsne_result, columns=["tsne_1", "tsne_2"])
tsne_df["cluster_id"] = clus["td_clusterid_kmeans"]
tsne_df["complaint_id"] = clus["consumer_complaint_narrative_tdf"]

In [ ]:
# Build an enriched DataFrame for the interactive scatter plot
HOVER_MAX_CHARS = 50

tsne_complaint_df = pd.DataFrame(tsne_result, columns=["tsne_1", "tsne_2"])
tsne_complaint_df["cluster_id"] = clus["td_clusterid_kmeans"]
tsne_complaint_df["complaint_id"] = clus["complaint_id_tdf"]
tsne_complaint_df["complaint"] = clus["consumer_complaint_narrative_tdf"]

# Truncate long complaint text for hover tooltips
tsne_complaint_df["truncated_complaint"] = clus[
    "consumer_complaint_narrative_tdf"
].apply(lambda x: x[:HOVER_MAX_CHARS] + "..." if len(x) > HOVER_MAX_CHARS else x)

# Interactive scatter plot of t-SNE clusters
fig = px.scatter(
    tsne_complaint_df,
    x="tsne_1",
    y="tsne_2",
    color="cluster_id",
    hover_data=["complaint_id", "truncated_complaint", "cluster_id"],
)
fig.update_traces(marker=dict(size=15))
fig.update_layout(
    title="t-SNE Visualization of Clusters with Complaints",
    xaxis_title="Dimension 1",
    yaxis_title="Dimension 2",
    xaxis=dict(tickangle=45),
    width=1000,
    height=800,
    hoverlabel=dict(bgcolor="white", font_size=16, font_family="Rockwell"),
    autosize=False,
)
fig.update_traces(
    hovertemplate=(
        "<b>Complaint ID:</b> %{customdata[0]}<br>"
        "<b>Complaint:</b> %{customdata[1]}<br>"
        "<b>Cluster ID:</b> %{customdata[2]}<br><extra></extra>"
    )
)
fig.show()

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial;'>8. Cleanup</b>

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial;'><b>8.1 Remove the Container</b></p>
<p style = 'font-size:16px;font-family:Arial;'>Remove the container if desired</p>

In [ ]:
# remove_env(OAF_NAME)

<footer style="padding-bottom:35px; background:#f9f9f9; border-bottom:3px solid #00233C">
    <div style="float:left;margin-top:14px">ClearScape Analytics™</div>
    <div style="float:right;">
        <div style="float:left; margin-top:14px">
            Copyright © Teradata Corporation - 2024. All Rights Reserved
        </div>
    </div>
</footer>